# 182 — Feature Composition Analysis

Analyze how features compose across layers: amplification tracking,
cancellation detection, cross-layer interaction, component alignment,
and composition scores.

## Why This Matters

Composition feature measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.feature_composition_analysis import (
    feature_amplification,
    feature_cancellation,
    cross_layer_feature_interaction,
    component_feature_alignment,
    feature_composition_scores,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(0), (32,))
result = feature_amplification(model, tokens, direction)
print(f"Total amplification: {result['total_amplification']:.3f}\n")
for s in result['stages']:
    print(f"Layer {s['layer']}: pre={s['pre_projection']:.4f}  post={s['post_projection']:.4f}  "
          f"attn={s['attn_contribution']:.4f}  mlp={s['mlp_contribution']:.4f}  "
          f"amp={s['amplification']:.2f}x")

In [ ]:
result = feature_cancellation(model, tokens)
print(f"Mean cancellation: {result['mean_cancellation']:.3f}\n")
for layer in result['per_layer']:
    bar = '#' * int(abs(layer['cancellation_ratio']) * 30)
    print(f"Layer {layer['layer']}: cos={layer['mean_cosine']:.3f}  "
          f"cancel_frac={layer['cancellation_fraction']:.3f}  "
          f"cancel_ratio={layer['cancellation_ratio']:.3f}  {bar}")

In [ ]:
for la in range(3):
    for lb in range(la+1, 4):
        result = cross_layer_feature_interaction(model, tokens, la, lb)
        print(f"L{la}->L{lb}: overlap={result['subspace_overlap']:.3f}  "
              f"mean_angle={result['mean_angle']:.3f}  "
              f"dir_cos={result['mean_direction_cosine']:.3f}")

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(1), (32,))
for layer in range(4):
    result = component_feature_alignment(model, tokens, layer=layer, direction=direction)
    print(f"Layer {layer}: attn={result['attn_mean_projection']:.4f}  "
          f"mlp={result['mlp_mean_projection']:.4f}  dominant={result['dominant_component']}")
    for h in result['per_head']:
        print(f"  Head {h['head']}: proj={h['mean_projection']:.4f}")

In [ ]:
result = feature_composition_scores(model, tokens)
print(f"Mean reinforcement: {result['mean_reinforcement']:.3f}\n")
for layer in result['per_layer']:
    kind = 'REINFORCE' if layer['is_reinforcing'] else 'CORRECT'
    print(f"Layer {layer['layer']}: delta/pre={layer['delta_to_pre_ratio']:.3f}  "
          f"reinforce={layer['reinforcement_score']:.3f}  {kind}")